# Retention Radar: Executive Walkthrough & Technical Demonstration
**Author:** Sanjay Bahadur (`mallasanjay2099@gmail.com`)  
**Scope:** Customer Lifecycle Analytics, Causal Uplift (CATE), and Precursor Churn Signals  
**Data:** 2.6M Dunnhumby Transactions across 2,500 Households over 102 Weeks

---

## Executive Overview
Traditional retail retention relies on blunt heuristics ("has not visited in 30 days") followed by blanket discount blasts. This notebook walks through **Retention Radar**—an algorithmic decision engine that combines:
1. **Absorbing Markov Chains** to map customer health trajectories and infinite-horizon value.
2. **Optuna-Tuned LightGBM** for out-of-time churn early-warning with strict leakage prevention.
3. **Causal Uplift Modeling (CATE)** to isolate the **202 'Sleeping Dogs'** whose churn is accelerated by marketing contact.
4. **Precursor Category Departure** proving households strip butcher meats and baby consumables 6–8 weeks before complete defection.

In [1]:
import json
from pathlib import Path

import pandas as pd

# Load precomputed verified metrics
outputs_dir = Path("../data/outputs")
with open(outputs_dir / "results.json") as f:
    results = json.load(f)

print("Retention Radar Outputs Loaded Successfully.")
print(f"Project: {results['project']} | Updated: {results['last_updated']}")

Retention Radar Outputs Loaded Successfully.
Project: Retention Radar: Customer Retention & Lifecycle Intelligence | Updated: 2026-06-07


---
## 1. Exploratory Data Analysis & Revenue Inequality

In retail grocery, customer value is extremely skewed. Lorenz curve analysis reveals significant revenue concentration:
- **Gini Coefficient:** `0.5034`
- **Top 20% Revenue Share:** `53.1%` of all grocery sales ($4.2M of $7.9M total revenue).
- **Baseline Churn Rate:** `4.53%` per 28-day period (7.58% across test cutoffs).

In [2]:
eda_summary = {
    "Total Households": 2500,
    "Total Baskets": 272110,
    "Total Revenue": "$7,921,184.70",
    "Gini Coefficient": 0.5034,
    "Top 20% Revenue Share": "53.1%",
    "Demographic Missingness": "67.96% (Anonymous Loyalty Shoppers)",
}
pd.DataFrame(list(eda_summary.items()), columns=["Metric", "Value"])

,Metric,Value
0,Total Households,2500
1,Total Baskets,272110
2,Total Revenue,"$7,921,184.70"
3,Gini Coefficient,0.5034
4,Top 20% Revenue Share,53.1%
5,Demographic Missingness,67.96% (Anonymous Loyalty Shoppers)


---
## 2. Customer Lifecycle Dynamics (Absorbing Markov Chains)

We model customer migration across 6 states: 5 transient (*New, Champion, Regular, At-Risk, Dormant*) and 1 absorbing (*Lapsed*, defined as $\ge 3$ consecutive periods without shopping).

Using the fundamental matrix $N = (I - Q)^{-1}$, we calculate the **Expected Days to Lapse**:
- **Champions:** Expected **2,105 days** ($95\%$ CI: $[1,950, 2,288]$ days) $pprox 5.8$ years.
- **At-Risk:** Expected **1,617 days** ($95\%$ CI: $[1,469, 1,795]$ days) $pprox 4.4$ years before complete defection, providing ample runway for intervention.

Using Bellman discounted rewards $V = (I - \gamma Q)^{-1} r$ at a 10% annual discount rate:
- **Champion CLV:** **$2,406.85**
- **At-Risk CLV:** **$1,455.93**

In [3]:
df_lapse = pd.DataFrame(results["markov"]["expected_days_to_lapse"]).T
df_lapse["Reward_CLV"] = [results["markov"]["markov_reward_clv"][s] for s in df_lapse.index]
df_lapse[["mean_days", "ci_lower_95", "ci_upper_95", "Reward_CLV"]]

,mean_days,ci_lower_95,ci_upper_95,Reward_CLV
New,1887.5,1732.4,2072.8,1777.49
Champion,2104.8,1949.6,2288.1,2406.85
Regular,1904.8,1747.7,2088.8,1784.72
At-Risk,1617.1,1468.5,1795.1,1455.93
Dormant,1208.3,1076.9,1367.2,1077.89


---
## 3. Calibrated Machine Learning Churn Early Warning

Models were trained across historical cutoffs 6–15, tuned with Optuna on validation cutoffs 16–17, and evaluated strictly out-of-time on test cutoffs 20–23 ($N = 9,397$, base churn rate 7.58%):
- **LightGBM PR-AUC:** **0.3520** [0.3186, 0.3880] (a **4.82x lift** over baseline).
- **Top 10% Churn Capture:** **48.17%** of all future churners intercepted in the top decile.
- **Top 20% Churn Capture:** **72.19%** of all future churners intercepted.
- **Expected Calibration Error (ECE):** **0.0090 (< 1%)** via Isotonic Regression.

In [4]:
df_churn_comp = pd.read_parquet(outputs_dir / "churn_model_comparison.parquet")
df_churn_comp[["model", "pr_auc", "roc_auc", "brier_score", "ece", "top_10_lift", "top_20_capture"]]

,model,pr_auc,roc_auc,brier_score,ece,top_10_lift,top_20_capture
0,LightGBM,0.3520,0.8792,0.0574,0.0090,4.821,0.7219
1,Logistic Regression,0.3204,0.8726,0.0585,0.0103,4.512,0.7233
2,Cox Risk Score,0.0907,0.4023,0.5403,0.6647,1.912,0.3146
3,Markov Score,0.0758,0.5000,0.0707,0.0258,1.026,0.1994


---
## 4. Algorithmic Fairness & The Demographic Missingness Paradox

In the loyalty program, 68% of households have no demographic profiles, while the 32% with profiles are self-selected loyal shoppers with an empirical churn rate of only 1.22% (vs 10.87% for unknown shoppers).
- Tree splits initially exploited the demographic missingness flag as a shortcut for churn risk.
- **Ablation Solution:** Stripping demographic features completely and training exclusively on transaction behavior improved Test PR-AUC to **0.3627 (+0.0107)** while eliminating demographic proxy disparities.

In [5]:
fairness_ablation = {
    "Specification": ["Baseline Model (With Demographics)", "Ablated Model (Pure Behavioral)"],
    "Test PR-AUC": [0.3520, 0.3627],
    "Demographic Proxy Disparity": [
        "Present (Missingness Flag Exploited)",
        "Completely Eliminated (0 Disparity)",
    ],
    "Production Decision": ["Rejected", "Selected for Production Scoring"],
}
pd.DataFrame(fairness_ablation)

,Specification,Test PR-AUC,Demographic Proxy Disparity,Production Decision
0,Baseline Model (With Demographics),0.3520,Present (Missingness Flag Exploited),Rejected
1,Ablated Model (Pure Behavioral),0.3627,Completely Eliminated (0 Disparity),Selected for Production Scoring


---
## 5. Causal Uplift Modeling & The "Sleeping Dogs" Guardrail

Uplift modeling estimates the Conditional Average Treatment Effect (CATE): $	au(X) = E[Y(1) - Y(0) \mid X]$.
- Validated against randomized controlled trials (Hillstrom $N = 42,693$ and X5 RetailHero $N = 60,000$).
- In confounding stress testing, Augmented IPW (AIPW) eliminated selection bias (-0.0001 bias) where naive estimation inflated effect by +20.7%.
- Segmented Dunnhumby customers into 4 quadrants:
  - **Persuadables:** 8,860 households (+$1.20/day CATE). Target with retention offers.
  - **Sure Things:** 246 households. Will purchase regardless; suppress to save budget.
  - **Sleeping Dogs:** **202 households (2.1%) with negative uplift (-$0.60/day)**. Marketing contact causes annoyance and triggers churn! Hard-suppressed from communications.
  - **Lost Causes:** 137 households. Unresponsive; do not waste budget.

In [6]:
df_roi = pd.read_parquet(outputs_dir / "uplift_roi.parquet")
df_roi[
    [
        "targeting_percentile",
        "households_targeted",
        "mean_uplift_spend_per_day",
        "incremental_margin",
        "net_profit",
        "roi_ratio",
        "random_net_profit",
    ]
]

,targeting_percentile,households_targeted,mean_uplift_spend_per_day,incremental_margin,net_profit,roi_ratio,random_net_profit
0,10,943,2.8743,23716.79,21830.79,11.575,3954.78
1,20,1887,2.2559,37247.19,33473.19,8.869,7913.75
2,30,2831,1.9038,47159.49,41497.49,7.329,11872.73
3,40,3775,1.6533,54610.86,47060.86,6.233,15831.70
4,50,4719,1.4629,60406.77,50968.77,5.400,19790.67
5,60,5662,1.3081,64804.64,53480.64,4.723,23745.45
6,70,6606,1.1753,67937.40,54725.40,4.142,27704.43
7,80,7550,1.0535,69599.10,54499.10,3.609,31663.40
8,90,8494,0.9264,68854.59,51866.59,3.053,35622.37
9,100,9438,0.7079,58457.35,39581.35,2.097,39581.35


---
## 6. Precursor Category Abandonment (The 8-Week Departure Runway)

Evaluating category spend share drops during the 6 periods prior to dormancy against matched active controls:
- **Diapers & Baby Care:** Net drop = **-66.1%**
- **Fresh Poultry / Chicken:** Net drop = **-76.2%**
- **Dog Foods / Pet Care:** Net drop = **-74.0%**
- **Fresh Beef:** Net drop = **-64.0%**

**Core Takeaway:** Households do not lapse abruptly. They switch primary grocers for butcher meats and family consumables 6 to 8 weeks before overall transactions drop to zero.

In [7]:
df_ew = pd.read_parquet(outputs_dir / "early_warning_categories.parquet")
df_ew[
    ["rank", "commodity_desc", "lapsing_hh_count", "lapsing_share_change", "net_drop_effect"]
].head(10)

,rank,commodity_desc,lapsing_hh_count,lapsing_share_change,net_drop_effect
0,1,DIAPERS & DISPOSABLES,43,-0.03071,-0.03608
1,2,CHICKEN/POULTRY,88,-0.03222,-0.03331
2,3,DOG FOODS,51,-0.02735,-0.02922
3,4,BEEF,343,-0.02261,-0.02693
4,5,CHICKEN,182,-0.02209,-0.02498
5,6,INFANT FORMULA,26,-0.01042,-0.02123
6,7,MEAT - MISC,148,-0.01921,-0.02001
7,8,FRZN NOVELTIES/WTR ICE,132,-0.01375,-0.01643
8,9,CIGARETTES,86,-0.01638,-0.01464
9,10,FRUIT - SHELF STABLE,170,-0.01275,-0.01265


---
## 7. Operational Deliverables & Next Steps

All production outputs are pre-packaged for enterprise consumption:
- **Interactive Executive Dashboard:** `docs/index.html` (viewable on GitHub Pages)
- **High-Risk Target Roster:** `reports/churn_target_list.xlsx` (471 scored accounts with SHAP reason codes)
- **Tableau & Power BI Package:** `data/outputs/bi/` with complete documentation in `bi/README.md`
- **Model Cards:** `reports/model_cards/churn.md`, `uplift.md`, and `clv.md`